# LegalAI — Integration Testing Notebook

End-to-end testing of the LegalAI system.

In [ ]:
import requests
import json

API_BASE = 'http://localhost:8000/api/v1'

# Health check
response = requests.get(f'{API_BASE}/health')
print(f'Health check: {response.json()}')

## 1. Test Document Upload

In [ ]:
# Create a test legal document
test_text = """IN THE SUPREME COURT OF INDIA
CRIMINAL APPEAL NO. 123 OF 2023

State of Maharashtra ... Appellant
vs.
Rajesh Kumar ... Respondent

JUDGMENT

The appellant, State of Maharashtra, has filed this appeal against the 
acquittal of the respondent by the Bombay High Court. The respondent was 
charged under Section 302 of the Indian Penal Code for the murder of 
Priya Sharma on the night of 15th August 2022 at Mumbai.

The prosecution evidence included DNA report, CCTV footage from the 
crime scene, and the testimony of three eyewitnesses. The post-mortem 
report confirmed that the cause of death was multiple stab wounds.

The accused also faces charges under Section 376 IPC for rape and 
Section 201 IPC for causing disappearance of evidence.

We have considered the evidence carefully. The DNA evidence conclusively 
links the respondent to the crime. The CCTV footage shows the respondent 
at the scene of the crime at the relevant time.

We set aside the acquittal and convict the respondent under Section 302 
IPC and Section 376 IPC. The respondent is sentenced to life imprisonment 
for the remainder of his natural life.

Date: 20th December 2023
Place: New Delhi
"""

# Save as test file
with open('/tmp/test_judgment.txt', 'w') as f:
    f.write(test_text)

# Upload
with open('/tmp/test_judgment.txt', 'rb') as f:
    response = requests.post(f'{API_BASE}/upload/', files={'file': ('test_judgment.txt', f, 'text/plain')})

upload_result = response.json()
doc_id = upload_result['id']
print(f'Upload result: {json.dumps(upload_result, indent=2)}')

## 2. Wait for Processing & Check Status

In [ ]:
import time

# Poll for completion
for i in range(30):
    response = requests.get(f'{API_BASE}/upload/{doc_id}')
    doc = response.json()
    status = doc['status']
    print(f'Attempt {i+1}: Status = {status}')
    if status in ['completed', 'failed']:
        break
    time.sleep(2)

print(f'\nFinal status: {doc["status"]}')
if doc.get('extracted_text'):
    print(f'Text length: {len(doc["extracted_text"])} chars')
if doc.get('metadata'):
    print(f'Case Number: {doc["metadata"].get("case_number")}')
    print(f'Court: {doc["metadata"].get("court_name")}')
    print(f'Crime Category: {doc["metadata"].get("crime_category")}')
    print(f'Parties: {doc["metadata"].get("parties")}')
if doc.get('entities'):
    print(f'Entities found: {len(doc["entities"])}')

## 3. Test Section Prediction

In [ ]:
response = requests.get(f'{API_BASE}/sections/predict/{doc_id}')
sections = response.json()

print(f'Total sections found: {sections["total_sections_found"]}')
print(f'Acts referenced: {sections["acts_referenced"]}')
print()

for ps in sections['predicted_sections']:
    sec = ps['section']
    print(f'[{ps["confidence"]:.0%}] {sec["act"]} Section {sec["section_number"]}: {sec["section_title"]}')
    if sec.get('punishment'):
        p = sec['punishment']
        print(f'       Punishment: {p["punishment_type"]} | Max: {p.get("maximum_duration", "N/A")} | Bailable: {p.get("is_bailable", "N/A")}')
    print()

## 4. Test Summarization

In [ ]:
response = requests.get(f'{API_BASE}/summary/{doc_id}')
summary = response.json()

print('SHORT SUMMARY:')
print(summary['short_summary'])
print()
print('KEY FINDINGS:')
for i, finding in enumerate(summary['key_findings'], 1):
    print(f'  {i}. {finding}')
print()
print('FINAL VERDICT:')
print(summary['final_verdict'])

## 5. Test Similar Cases

In [ ]:
response = requests.get(f'{API_BASE}/cases/similar/{doc_id}?top_k=5')
cases = response.json()

print(f'Found {cases["total_results"]} similar cases:\n')

for case in cases['similar_cases']:
    print(f'{case["case_title"]}')
    print(f'  Court: {case["court"]} | Match: {case["similarity_score"]:.0%} | Outcome: {case["outcome"]}')
    print(f'  Sections: {", ".join(case["acts_sections"])}')
    print()

## 6. Test Chat

In [ ]:
response = requests.post(f'{API_BASE}/chat/', json={
    'message': 'Which laws apply to this case?',
    'document_id': doc_id,
    'chat_history': []
})

chat_result = response.json()
print(f'Answer: {chat_result["message"]}')
print(f'Context used: {chat_result["context_used"]} sources')

## 7. Test Section Explanation

In [ ]:
response = requests.get(f'{API_BASE}/sections/explain/IPC/302')
explanation = response.json()

print(f'Section: {explanation["section_title"]}')
print(f'\nSimple Explanation:')
print(explanation['simple_explanation'])
print(f'\nWhen Applies:')
print(explanation['when_applies'])
print(f'\nExample Cases:')
for case in explanation['example_cases']:
    print(f'  • {case}')